# Unit 2 — Taxi-v3: Q-Learning from scratch

**Pass criterion:** `mean_reward − std_reward ≥ 4`

No RL library this time: **we write the Q-table and the update rule ourselves**, using only `numpy`.

**How to run:** `Runtime → Run all` (~3–5 min, no GPU). The Hugging Face login reads the `HF_TOKEN` Colab secret.

In [ ]:
!pip install -q "gymnasium[toy-text]==0.29.1" huggingface_hub imageio imageio-ffmpeg tqdm

In [ ]:
HF_USERNAME = "Zorlu5454"   # <- your Hugging Face username
ENV_ID = "Taxi-v3"
REPO_ID = f"{HF_USERNAME}/q-{ENV_ID}"

import base64, datetime, json, pickle, random, tempfile
from pathlib import Path
import imageio
import numpy as np
import gymnasium as gym
from tqdm.auto import tqdm
from IPython.display import HTML, display

## 1. The environment
- A 5×5 city with 4 stops (R, G, Y, B). A passenger waits at one stop and wants to go to another.
- **500 states** → 25 taxi positions × 5 passenger locations (4 stops + inside the taxi) × 4 destinations.
- **6 actions** → south, north, east, west, pick up, drop off.
- **Rewards (built into the game):** −1 per step, +20 for a correct drop-off, −10 for an illegal pick-up / drop-off.

In [ ]:
env = gym.make(ENV_ID, render_mode="rgb_array")
state_space = env.observation_space.n
action_space = env.action_space.n
print("States :", state_space)
print("Actions:", action_space)

## 2. The Q-table
500 rows (states) × 6 columns (actions). All zeros at the start: the agent knows nothing.

In [ ]:
def initialize_q_table(state_space, action_space):
    return np.zeros((state_space, action_space))

Qtable = initialize_q_table(state_space, action_space)
print(Qtable.shape)

## 3. Action selection
- **Greedy:** look up the row, pick the action with the largest value → *exploitation*
- **ε-greedy:** random action with probability ε, otherwise greedy → *exploration + exploitation*

In [ ]:
def greedy_policy(Qtable, state):
    return int(np.argmax(Qtable[state]))

def epsilon_greedy_policy(Qtable, state, epsilon):
    if random.uniform(0, 1) > epsilon:
        return greedy_policy(Qtable, state)   # exploit: use what we know
    return env.action_space.sample()          # explore: try something random

## 4. Hyperparameters
- `learning_rate=0.7` (α): how far the estimate is moved towards the target.
- `gamma=0.95` (γ): discount factor — how much the future matters.
- ε decays exponentially from **1.0** to **0.05**: explore a lot first, then trust the table.
- `eval_seed`: **do not change** — the 100 fixed start states defined by the course so that scores are comparable.

In [ ]:
n_training_episodes = 25000
learning_rate = 0.7
n_eval_episodes = 100
max_steps = 99
gamma = 0.95
max_epsilon = 1.0
min_epsilon = 0.05
decay_rate = 0.005

# DO NOT MODIFY: fixed evaluation starting states used by the course
eval_seed = [16,54,165,177,191,191,120,80,149,178,48,38,6,125,174,73,50,172,100,148,146,6,25,40,68,148,49,167,9,97,164,176,61,7,54,55,
 161,131,184,51,170,12,120,113,95,126,51,98,36,135,54,82,45,95,89,59,95,124,9,113,58,85,51,134,121,169,105,21,30,11,50,65,12,43,82,145,152,97,106,55,31,85,38,
 112,102,168,123,97,21,83,158,26,80,63,5,81,32,11,28,148]

## 5. Video helper
Plays one episode and shows it as a video. With an empty Q-table (all zeros) the taxi always picks the same action and never arrives.

In [ ]:
def show_agent(Qtable, seed=16, name="taxi", fps=3):
    env_v = gym.make(ENV_ID, render_mode="rgb_array")
    state, _ = env_v.reset(seed=seed)
    frames, total = [env_v.render()], 0
    for _ in range(max_steps):
        action = greedy_policy(Qtable, state)
        state, reward, terminated, truncated, _ = env_v.step(action)
        total += reward
        frames.append(env_v.render())
        if terminated or truncated:
            break
    env_v.close()
    path = f"{name}.mp4"
    imageio.mimsave(path, frames, fps=fps, macro_block_size=1)
    print(f"Episode reward: {total}  ({len(frames) - 1} steps)")
    b64 = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="400" controls autoplay loop>'
                 f'<source src="data:video/mp4;base64,{b64}" type="video/mp4"></video>'))

show_agent(Qtable, name="untrained_taxi")

## 6. Training loop (where the learning happens)
At every step:
1. Choose an action with ε-greedy.
2. Apply it → observe the reward and the next state.
3. **Q-learning update:**
   `Q(s,a) ← Q(s,a) + α · [ r + γ·max_a' Q(s',a') − Q(s,a) ]`
   In words: *"move my estimate a little towards (reward received + value of the best action in the next state)."*

In [ ]:
def train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable):
    for episode in tqdm(range(n_training_episodes)):
        # Less exploration over time
        epsilon = min_epsilon + (max_epsilon - min_epsilon) * np.exp(-decay_rate * episode)
        state, _ = env.reset()
        for _ in range(max_steps):
            action = epsilon_greedy_policy(Qtable, state, epsilon)
            new_state, reward, terminated, truncated, _ = env.step(action)

            # Q-learning update
            target = reward + gamma * np.max(Qtable[new_state])
            Qtable[state][action] += learning_rate * (target - Qtable[state][action])

            if terminated or truncated:
                break
            state = new_state
    return Qtable

Qtable = train(n_training_episodes, min_epsilon, max_epsilon, decay_rate, env, max_steps, Qtable)

## 7. Inspect the table
Everything was 0 before training. Looking at a state's row now shows which action the agent picks and why.

In [ ]:
actions = ["South", "North", "East", "West", "Pickup", "Dropoff"]
state, _ = env.reset(seed=16)
print("State:", state)
for a, q in zip(actions, Qtable[state]):
    print(f"  {a:8s} Q = {q:7.2f}")
print("Best action ->", actions[greedy_policy(Qtable, state)])

## 8. Evaluate (on the 100 fixed start states)

In [ ]:
def evaluate_agent(env, max_steps, n_eval_episodes, Q, seed):
    episode_rewards = []
    for episode in range(n_eval_episodes):
        state, _ = env.reset(seed=seed[episode]) if seed else env.reset()
        total = 0
        for _ in range(max_steps):
            action = greedy_policy(Q, state)
            state, reward, terminated, truncated, _ = env.step(action)
            total += reward
            if terminated or truncated:
                break
        episode_rewards.append(total)
    return np.mean(episode_rewards), np.std(episode_rewards)

mean_reward, std_reward = evaluate_agent(env, max_steps, n_eval_episodes, Qtable, eval_seed)
score = mean_reward - std_reward
print(f"mean_reward = {mean_reward:.2f} +/- {std_reward:.2f}")
print(f"score (mean - std) = {score:.2f}  ->  {'PASSED ✅' if score >= 4 else 'NOT YET ❌'}")

## 9. Watch the trained taxi

In [ ]:
show_agent(Qtable, seed=16, name="trained_taxi_1")
show_agent(Qtable, seed=54, name="trained_taxi_2")

## 10. Push to the Hugging Face Hub
The login cell reads the `HF_TOKEN` Colab secret (**Write** token).

The model card gets the `Taxi-v3` and `q-learning` tags, which the certification checker looks for.

In [ ]:
import os
from huggingface_hub import login, notebook_login
try:
    from google.colab import userdata
    tok = userdata.get("HF_TOKEN")        # Colab secret (🔑 left panel)
    os.environ["HF_TOKEN"] = tok
    login(token=tok, add_to_git_credential=False)
    print("Logged in with Colab secret HF_TOKEN")
except Exception as e:
    print("No Colab secret, falling back to interactive login:", e)
    notebook_login()

In [ ]:
from huggingface_hub import HfApi
from huggingface_hub.repocard import metadata_eval_result, metadata_save

def push_to_hub(repo_id, model, env):
    api = HfApi()
    api.create_repo(repo_id=repo_id, exist_ok=True)
    repo_name = repo_id.split("/")[1]
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        with open(tmp / "q-learning.pkl", "wb") as f:
            pickle.dump(model, f)

        mean_reward, std_reward = evaluate_agent(
            env, model["max_steps"], model["n_eval_episodes"], model["qtable"], model["eval_seed"])
        json.dump({"env_id": model["env_id"], "mean_reward": mean_reward,
                   "n_eval_episodes": model["n_eval_episodes"],
                   "eval_datetime": datetime.datetime.now().isoformat()},
                  open(tmp / "results.json", "w"))

        metadata = {"tags": [model["env_id"], "q-learning", "reinforcement-learning", "custom-implementation"]}
        metadata.update(metadata_eval_result(
            model_pretty_name=repo_name, task_pretty_name="reinforcement-learning",
            task_id="reinforcement-learning", metrics_pretty_name="mean_reward",
            metrics_id="mean_reward", metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
            dataset_pretty_name=model["env_id"], dataset_id=model["env_id"]))

        readme = tmp / "README.md"
        readme.write_text(f"""
# **Q-Learning** Agent playing **{model['env_id']}**
This is a trained model of a **Q-Learning** agent playing **{model['env_id']}**.

## Usage
```python
model = load_from_hub(repo_id="{repo_id}", filename="q-learning.pkl")
env = gym.make(model["env_id"])
```
""", encoding="utf-8")
        metadata_save(readme, metadata)

        # Replay video
        env_v = gym.make(model["env_id"], render_mode="rgb_array")
        state, _ = env_v.reset(seed=random.randint(0, 500))
        frames = [env_v.render()]
        for _ in range(model["max_steps"]):
            state, _, terminated, truncated, _ = env_v.step(greedy_policy(model["qtable"], state))
            frames.append(env_v.render())
            if terminated or truncated:
                break
        imageio.mimsave(tmp / "replay.mp4", frames, fps=1, macro_block_size=1)

        api.upload_folder(repo_id=repo_id, folder_path=tmp, path_in_repo=".")
    print(f"Done: https://huggingface.co/{repo_id}  ({mean_reward:.2f} +/- {std_reward:.2f})")

model = {
    "env_id": ENV_ID, "max_steps": max_steps, "n_training_episodes": n_training_episodes,
    "n_eval_episodes": n_eval_episodes, "eval_seed": eval_seed,
    "learning_rate": learning_rate, "gamma": gamma,
    "max_epsilon": max_epsilon, "min_epsilon": min_epsilon, "decay_rate": decay_rate,
    "qtable": Qtable,
}
push_to_hub(REPO_ID, model, env)

## 11. Check
https://huggingface.co/spaces/ThomasSimonini/Check-my-progress-Deep-RL-Course → **Unit 2** should be green ✅

In [ ]:
from huggingface_hub import HfApi, ModelCard

def check_like_certificate(env_tag, lib_tag, min_result):
    """Same query the course's Check-my-progress Space runs."""
    api = HfApi()
    ids = [m.id for m in api.list_models(author=HF_USERNAME,
                                         filter=["reinforcement-learning", env_tag, lib_tag])]
    if not ids:
        print(f"❌ No model with tags [{env_tag}, {lib_tag}] under {HF_USERNAME} (wait ~1 min and retry)")
        return
    for mid in ids:
        try:
            meta = ModelCard.load(mid).data.to_dict()
            value = str(meta["model-index"][0]["results"][0]["metrics"][0]["value"])
            if "+/-" in value:
                mean, std = (float(x) for x in value.split("+/-"))
            else:
                mean, std = float(value), 0.0
            result = mean - std
            print(f"{'✅' if result >= min_result else '❌'} {mid}: {result:.2f} (needs >= {min_result})")
        except Exception:
            print(f"✅ {mid}: found (no score in card; this unit only needs the upload)")

check_like_certificate("Taxi-v3", "q-learning", 4)